# Pre-processing

In [15]:
import json

import pandas as pd
from pathlib import Path

RESULTS_DIR = Path('')# gift eval results folder

EXCLUDE_FOLDERS = {'GRAIN'}

TARGET_DATASETS = ['m4_yearly']

TARGET_COLUMNS = [
    'eval_metrics/MASE[0.5]',
    'eval_metrics/sMAPE[0.5]',
    'eval_metrics/mean_weighted_sum_quantile_loss'
]

# Metadata pulled from each model folder's config.json
META_COLUMNS = ['model_type', 'testdata_leakage']

POSSIBLE_FILENAMES = ['all_results.csv', 'all-results.csv']

all_dfs = []

for folder in sorted(RESULTS_DIR.iterdir()):
    if not folder.is_dir():
        continue

    if folder.name in EXCLUDE_FOLDERS:
        print(f'Skipping: {folder.name}')
        continue

    # Find which CSV exists
    csv_path = None
    for fname in POSSIBLE_FILENAMES:
        candidate = folder / fname
        if candidate.exists():
            csv_path = candidate
            break

    if csv_path is None:
        print(f'No matching CSV in: {folder.name}')
        continue

    df = pd.read_csv(csv_path)

    # Identify dataset column
    dataset_col = None
    for candidate in ['dataset', 'item_id', 'name', 'dataset_name']:
        if candidate in df.columns:
            dataset_col = candidate
            break

    if dataset_col is None:
        print(f'Could not find dataset column in: {folder.name}. Columns: {df.columns.tolist()}')
        continue

    # Filter rows
    filtered = df[
        df[dataset_col]
        .astype(str)
        .str.split('/')
        .str[0]
        .isin(TARGET_DATASETS)
    ].copy()

    if filtered.empty:
        print(f'No matching rows in: {folder.name}')
        continue

    # Keep dataset col + target metric columns
    cols_to_keep = [dataset_col] + [c for c in TARGET_COLUMNS if c in df.columns]
    filtered = filtered[cols_to_keep].copy()
    filtered['model'] = folder.name

    # Attach metadata from config.json (missing file / missing keys -> NA)
    config_path = folder / 'config.json'
    config = {}
    if config_path.exists():
        try:
            with open(config_path) as fh:
                config = json.load(fh)
        except (json.JSONDecodeError, OSError) as e:
            print(f'Could not parse config.json in {folder.name}: {e}')
    else:
        print(f'No config.json in: {folder.name}')

    for col in META_COLUMNS:
        filtered[col] = config.get(col, pd.NA)

    all_dfs.append(filtered)
    print(f'Loaded {len(filtered)} rows from: {folder.name}')

if not all_dfs:
    print("No data collected.")
else:
    combined = pd.concat(all_dfs, ignore_index=True)
    print(combined.shape)
    print(combined.head())

Skipping: GRAIN
No data collected.


In [16]:
# Split by frequency from the dataset column
combined['freq'] = combined['dataset'].str.split('/').str[1]

yearly = combined[combined['freq'] == 'A'].drop(columns='freq').reset_index(drop=True)

yearly.to_csv('results_yearly.csv', index=False)

print(f'Yearly: {len(yearly)} rows')

Yearly: 1 rows


In [17]:
yearly_sorted = yearly.sort_values('eval_metrics/MASE[0.5]').reset_index(drop=True)


yearly_sorted.to_csv('results_yearly.csv', index=False)


print(yearly_sorted)


             dataset  eval_metrics/MASE[0.5]  eval_metrics/sMAPE[0.5]  \
0  m4_yearly/A/short                2.898559                 0.130911   

   eval_metrics/mean_weighted_sum_quantile_loss  model model_type  \
0                                      0.103408  GRAIN       <NA>   

  testdata_leakage  
0             <NA>  


In [18]:
# Load all GRAIN csvs
grain_yearly_df = pd.read_csv('GRAIN/all_results.csv')


# Yearly
grain_yearly_avg = grain_yearly_df.groupby('model')[['eval_metrics/MASE[0.5]', 'eval_metrics/sMAPE[0.5]', 'eval_metrics/mean_weighted_sum_quantile_loss']].mean().reset_index()

# GRAIN has no config.json, so declare their metadata here
GRAIN_META = {'model_type': 'zero-shot', 'testdata_leakage': 'No'}
for col in META_COLUMNS:
    grain_yearly_avg[col] = GRAIN_META.get(col, pd.NA)

yearly_final = pd.concat([yearly_sorted, grain_yearly_avg], ignore_index=True)
yearly_final = yearly_final.sort_values('eval_metrics/MASE[0.5]').reset_index(drop=True)

# Rank by MASE, drop dataset, reorder columns
yearly_final['rank'] = yearly_final['eval_metrics/MASE[0.5]'].rank(method='min').astype(int)
yearly_final = yearly_final[['rank', 'model'] + META_COLUMNS + TARGET_COLUMNS]

yearly_final.to_csv('results_yearly.csv', index=False)


print('Yearly:'); print(yearly_final)

Yearly:
   rank  model model_type testdata_leakage  eval_metrics/MASE[0.5]  \
0     1  GRAIN        NaN              NaN                2.898559   
1     1  GRAIN  zero-shot               No                2.898559   

   eval_metrics/sMAPE[0.5]  eval_metrics/mean_weighted_sum_quantile_loss  
0                 0.130911                                      0.103408  
1                 0.130911                                      0.103408  


In [19]:
# One CSV per metric, each ranked by that metric (lower is better)
OUTPUTS = {
    'eval_metrics/MASE[0.5]': 'results_yearly_mase.csv',
    'eval_metrics/sMAPE[0.5]': 'results_yearly_smape.csv',
    'eval_metrics/mean_weighted_sum_quantile_loss': 'results_yearly_quantile.csv',
}

for metric, out_path in OUTPUTS.items():
    df = yearly_final[['model'] + META_COLUMNS + [metric]].dropna(subset=[metric])
    df = df.sort_values(metric).reset_index(drop=True)
    df['rank'] = df[metric].rank(method='min').astype(int)
    df = df[['rank', 'model'] + META_COLUMNS + [metric]]
    df.to_csv(out_path, index=False)
    print(f'{out_path}: {len(df)} rows')
    print(df.head(), '\n')

results_yearly_mase.csv: 2 rows
   rank  model model_type testdata_leakage  eval_metrics/MASE[0.5]
0     1  GRAIN        NaN              NaN                2.898559
1     1  GRAIN  zero-shot               No                2.898559 

results_yearly_smape.csv: 2 rows
   rank  model model_type testdata_leakage  eval_metrics/sMAPE[0.5]
0     1  GRAIN        NaN              NaN                 0.130911
1     1  GRAIN  zero-shot               No                 0.130911 

results_yearly_quantile.csv: 2 rows
   rank  model model_type testdata_leakage  \
0     1  GRAIN        NaN              NaN   
1     1  GRAIN  zero-shot               No   

   eval_metrics/mean_weighted_sum_quantile_loss  
0                                      0.103408  
1                                      0.103408   



In [20]:
# Combined rankings: raw metric value + normalized (divided by seasonal_naive, so seasonal_naive = 1.0)
COMBINED_OUTPUTS = {
    'eval_metrics/MASE[0.5]': 'results_yearly_mase_normalized.csv',
    'eval_metrics/sMAPE[0.5]': 'results_yearly_smape_normalized.csv',
    'eval_metrics/mean_weighted_sum_quantile_loss': 'results_yearly_quantile_normalized.csv',
}

BASELINE = 'seasonal_naive'

for metric, out_path in COMBINED_OUTPUTS.items():
    df = yearly_final[['model'] + META_COLUMNS + [metric]].dropna(subset=[metric]).copy()

    baseline_val = df.loc[df['model'] == BASELINE, metric]
    if baseline_val.empty:
        print(f'WARNING: {BASELINE} not found for {metric}, skipping {out_path}')
        continue
    baseline_val = baseline_val.iloc[0]

    df['normalized'] = df[metric] / baseline_val
    df = df.sort_values(metric).reset_index(drop=True)
    df['rank'] = df[metric].rank(method='min').astype(int)
    df = df[['rank', 'model'] + META_COLUMNS + [metric, 'normalized']]
    df.to_csv(out_path, index=False)
    print(f'{out_path}: {len(df)} rows (baseline {BASELINE}={baseline_val:.4f})')
    print(df.head(), '\n')